In [3]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [4]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"

df_final_path = Path("./generated/df_final")
df_final_path.mkdir(exist_ok=True, parents=True)

In [ ]:
amazon_bounds = (
    gpd.read_file(initial_path / "AFP_fixed.gpkg")
    .assign(
        geometry=lambda df: df["geometry"].force_2d(),
    )
    .to_crs("ESRI:54009")["geometry"]
    .item()
)

In [ ]:
for YEAR in range(1975, 2021, 5):
    df_base = (
        gpd.read_file(
            generated_path / "polygons" / "population" / "200_300" / f"{YEAR}.gpkg",
        )
        .assign(
            combined_polygon_id=lambda df: [
                f"p{str(i).zfill(6)}" for i in range(len(df))
            ],
        )
        .set_index("combined_polygon_id")
    )

    max_idx = max(int(x[1:]) for x in df_base.index)
    df_modified = (
        gpd.read_file(
            generated_path / "polygons" / "population" / "150_200" / f"{YEAR}.gpkg",
        )
        .assign(
            combined_polygon_id=lambda df: [
                f"p{str(i + max_idx + 1).zfill(6)}" for i in range(len(df))
            ],
        )
        .set_index("combined_polygon_id")
    )

    df_modified_in_amazon = df_modified[df_modified.intersects(amazon_bounds)].copy()

    idx_base_redundant = (
        df_base[["geometry"]]
        .sjoin(
            df_modified_in_amazon[["geometry"]],
            how="inner",
            predicate="intersects",
        )
        .index.unique()
    )

    df_final = (
        pd.concat(
            [
                df_base.drop(index=idx_base_redundant).assign(in_amazon="no"),
                df_modified_in_amazon.assign(in_amazon="yes"),
            ],
            ignore_index=False,
        )
        .drop(columns=["polygon_id"])
        .reset_index(names="polygon_id")
        .drop(
            columns={
                f"pop{infix}_{year}"
                for infix in ["", "_urban_center", "_urban_cluster", "_rural"]
                for year in range(YEAR + 5, 2021, 5)
            },
        )
        .drop(
            columns={f"density_{year}" for year in range(1975, 2021, 5) if year > YEAR},
        )
        .pipe(lambda df: gpd.GeoDataFrame(df, geometry="geometry", crs=df.crs))
    )

    df_final.to_file(df_final_path / f"{YEAR}.gpkg", driver="GPKG")
    break

In [6]:
df_final

,polygon_id,GID_0,GID_1,GID_2,GID_3,GID_4,NAME_0,NAME_1,NAME_2,NAME_3,...,name,max_name,pop_1975,pop_urban_center_1975,pop_urban_cluster_1975,pop_rural_1975,area_km2,density_1975,geometry,in_amazon
0,p000000,MEX,MEX.3_1,MEX.3.2_2,None,None,México,Baja California,Mexicali,None,...,Los Algodones,Los Algodones,1798.787570,0.0,0.000000,1798.787570,3.0,599.595857,"POLYGON ((-10332000 3961000, -10332000 3960000...",no
1,p000001,MEX,MEX.3_1,MEX.3.2_2,None,None,México,Baja California,Mexicali,None,...,Ciudad Morelos (Cuervos),Ciudad Morelos (Cuervos),3071.681660,0.0,0.000000,3071.681660,5.0,614.336332,"POLYGON ((-10350000 3951000, -10350000 3950000...",no
2,p000002,MEX,MEX.3_1,MEX.3.2_2,None,None,México,Baja California,Mexicali,None,...,Islas Agrarias A,Islas Agrarias A,539.845545,0.0,0.000000,539.845545,2.0,269.922773,"POLYGON ((-10394000 3948000, -10394000 3947000...",no
3,p000003,MEX,MEX.3_1,MEX.3.2_2,None,None,México,Baja California,Mexicali,None,...,Poblado Paredones,Poblado Paredones,879.212386,0.0,0.000000,879.212386,2.0,439.606193,"POLYGON ((-10358000 3948000, -10358000 3947000...",no
4,p000004,MEX,MEX.3_1,MEX.3.2_2,None,None,México,Baja California,Mexicali,None,...,Benito Juárez,Benito Juárez,1443.557919,0.0,0.000000,1443.557919,3.0,481.185973,"POLYGON ((-10366000 3944000, -10366000 3943000...",no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34066,p067208,BOL,BOL.1_1,BOL.1.1_2,BOL.1.1.1_2,None,Bolivia,Chuquisaca,Azurduy,Azurduy,...,Cimientos,Cimientos,279.935362,0.0,0.000000,279.935362,1.0,279.935362,"POLYGON ((-6211000 -2459000, -6211000 -2460000...",yes
34067,p067223,BOL,BOL.1_1,BOL.1.3_2,BOL.1.3.2_2,None,Bolivia,Chuquisaca,Hernando Siles,Monteagudo,...,San Miguel del Bañado,San Miguel del Bañado,777.152578,0.0,0.000000,777.152578,2.0,388.576289,"POLYGON ((-6160000 -2462000, -6160000 -2464000...",yes
34068,p067248,BOL,BOL.1_1,BOL.1.1_2,BOL.1.1.1_2,None,Bolivia,Chuquisaca,Azurduy,Azurduy,...,Azurduy+Piedra Grande,Azurduy,6829.293270,0.0,6167.093535,662.199735,6.0,1138.215545,"POLYGON ((-6209000 -2465000, -6209000 -2466000...",yes
34069,p067386,BOL,BOL.1_1,BOL.1.3_2,BOL.1.3.2_2,None,Bolivia,Chuquisaca,Hernando Siles,Monteagudo,...,San Juan del Pirai,San Juan del Pirai,511.400023,0.0,0.000000,511.400023,2.0,255.700012,"POLYGON ((-6177000 -2493000, -6177000 -2494000...",yes
